# Linux Special Permissions: Sticky Bit, SUID, and SGID

This educational notebook explains Linux's three special permission bits with examples, safe laboratory commands, security warnings, review questions, and answers.

> **Safety warning:** Never experiment with SUID on real system programs such as `cat`, `bash`, Python, editors, or copy tools. Use a disposable virtual machine for privileged experiments.

## 1. Review: the 12 permission bits

A Linux mode is often described as **12 bits**:

- 9 ordinary bits: `rwx rwx rwx` for owner, group, and others
- 3 special bits: **SUID**, **SGID**, and **sticky bit**

| Special bit | Octal value | Main use |
|---|---:|---|
| SUID | `4000` | Run an executable with the file owner's effective UID |
| SGID | `2000` | Run with the file group's effective GID, or inherit group ownership in a directory |
| Sticky | `1000` | Restrict deletion and renaming inside a shared directory |

Examples:

```text
4755 = SUID + rwxr-xr-x
2755 = SGID + rwxr-xr-x
1777 = sticky + rwxrwxrwx
6755 = SUID + SGID + rwxr-xr-x
```

## 2. How special bits appear in `ls -l`

Special letters replace execute positions:

| Bit | Position |
|---|---|
| SUID | owner execute position |
| SGID | group execute position |
| Sticky | others execute position |

```text
-rwsr-xr-x  SUID set and owner execute set
-rwSr-xr-x  SUID set but owner execute missing
-rwxr-sr-x  SGID set and group execute set
-rwxr-Sr-x  SGID set but group execute missing
drwxrwxrwt  sticky set and others execute set
drwxrwxrwT  sticky set but others execute missing
```

Lowercase `s`/`t` means the special bit and corresponding execute bit are both present. Uppercase `S`/`T` means the special bit is present but execute is absent.

# Part I — Sticky Bit

## 3. What is the sticky bit?

The sticky bit is mainly useful on **directories**. In a writable sticky directory, a user cannot normally delete or rename another user's file merely because the directory is writable.

An entry may normally be removed or renamed only by:

1. the file owner,
2. the directory owner, or
3. root.

The sticky bit does not prevent the file owner from changing the file's contents. It controls deletion and renaming of directory entries.

## 4. Why is it needed?

A directory with mode `0777` is writable by everyone:

```text
drwxrwxrwx shared/
```

Without the sticky bit, one user may delete or rename another user's files because deletion is governed primarily by the **parent directory's** `w` and `x` permissions.

With mode `1777`:

```text
drwxrwxrwt shared/
```

users may create files, but normally cannot delete or rename files owned by other users.

## 5. Enable and disable the sticky bit

Symbolic form:

```bash
chmod +t directory
chmod -t directory
```

Numeric form:

```bash
chmod 1777 directory   # sticky + rwxrwxrwx
chmod 0777 directory   # remove sticky, retain rwxrwxrwx
```

The leading `1` represents the sticky bit.

In [ ]:
mkdir -p sticky_demo
chmod 0777 sticky_demo
ls -ld sticky_demo
chmod +t sticky_demo
ls -ld sticky_demo
stat -c 'symbolic=%A numeric=%a owner=%U group=%G name=%n' sticky_demo
chmod -t sticky_demo

## 6. Practical meaning of a sticky directory

Suppose a shared directory contains `alice.txt` owned by Alice and `bob.txt` owned by Bob.

- Alice may normally delete or rename `alice.txt`.
- Bob may normally delete or rename `bob.txt`.
- Alice may not normally delete or rename `bob.txt`.
- Bob may not normally delete or rename `alice.txt`.
- root and the directory owner may administer all entries.

This makes the sticky bit ideal for publicly writable directories.

## 7. Applications of the sticky bit

Common uses include:

- `/tmp`
- `/var/tmp`
- shared upload directories
- university or company multi-user servers
- shared scratch/build directories
- public drop areas

It is most useful when many users have write access to the same directory.

## 8. Why `/tmp` has the sticky bit

Most systems show something like:

```text
drwxrwxrwt root root /tmp
```

The usual numeric mode is `1777`:

- all users can enter `/tmp`,
- all users can create temporary files,
- users cannot normally delete or rename another user's entries,
- root and the directory owner retain administrative control.

Without the sticky bit, `/tmp` would be unsafe on a multi-user machine.

In [ ]:
ls -ld /tmp
stat -c '%A %a %U %G %n' /tmp

## 9. Should `/tmp` be a separate partition?

A separate filesystem for `/tmp` can be good practice on servers, but it is **not mandatory in every system**.

Possible benefits:

- temporary files cannot fill the root filesystem,
- size limits and quotas are easier,
- temporary I/O is isolated,
- separate mount options can be applied,
- temporary contents can be cleared independently.

Common mount options:

| Option | Meaning |
|---|---|
| `nodev` | Do not interpret device nodes |
| `nosuid` | Ignore SUID/SGID execution behavior |
| `noexec` | Prevent direct execution from that filesystem |
| `relatime` | Reduce access-time writes |

Cautions:

- `noexec` is not a complete security boundary.
- Installers, build tools, containers, and language runtimes may need to execute temporary helpers.
- Modern systems may use `tmpfs` rather than a disk partition.

## 10. Creating and mounting a separate `/tmp`

### Disk partition method

High-level procedure:

1. create/select a partition,
2. format it,
3. mount it temporarily,
4. set ownership and mode,
5. add it to `/etc/fstab`,
6. test before rebooting.

Example:

```bash
sudo mkfs.ext4 /dev/sdXN
sudo mount /dev/sdXN /mnt
sudo chown root:root /mnt
sudo chmod 1777 /mnt
sudo blkid /dev/sdXN
```

Example `/etc/fstab` entry:

```text
UUID=YOUR-UUID  /tmp  ext4  defaults,nodev,nosuid  0  2
```

Then test:

```bash
sudo mount -a
findmnt /tmp
ls -ld /tmp
```

An incorrect `/etc/fstab` entry can cause boot problems. Replace placeholders with actual values.

### `tmpfs` method

```text
tmpfs /tmp tmpfs defaults,nodev,nosuid,mode=1777,size=2G 0 0
```

Advantages: fast and cleared at reboot. Disadvantages: consumes RAM/swap and loses contents at reboot.

Some distributions manage `/tmp` through systemd:

```bash
systemctl status tmp.mount
```

# Part II — SUID

## 11. What is SUID?

**SUID** means **Set User ID**. It is meaningful mainly on executable binary files.

When a user runs a valid SUID executable, the process normally has:

- real UID = the user who launched it,
- effective UID = the executable file's owner.

Many permission checks use the effective UID. Therefore, a root-owned SUID executable can perform limited privileged operations for ordinary users.

## 12. Is SUID for a file, directory, or program?

The bit is stored on a filesystem object, but its useful effect is mainly on **compiled executable files**.

- On ordinary data files, it has no useful execution effect.
- On Linux directories, SUID is generally ignored.
- Linux normally ignores SUID on interpreted scripts for security reasons.
- A filesystem mounted with `nosuid` ignores SUID/SGID execution behavior.

## 13. Enable, disable, and inspect SUID

```bash
chmod u+s program      # enable
chmod u-s program      # disable
chmod 4755 program     # numeric enable
chmod 0755 program     # numeric disable while retaining 755
```

Inspect it:

```bash
ls -l program
stat -c '%A %a %U %G %n' program
```

`-rwsr-xr-x` means SUID plus owner execute. `-rwSr-xr-x` means SUID is set but owner execute is missing.

## 14. What happens with `cat /etc/shadow`?

An ordinary user typically receives:

```text
cat: /etc/shadow: Permission denied
```

`/etc/shadow` stores sensitive password-related account data. Typical ownership and permissions resemble:

```text
-rw-r----- root shadow /etc/shadow
```

An ordinary user is not root and usually is not in the restricted `shadow` group, so the kernel denies reading it.

In [ ]:
ls -l /etc/shadow
stat -c '%A %a %U %G %n' /etc/shadow

## 15. Why `sudo chmod u+s /bin/cat` is dangerous

Do **not** run this on a real system:

```bash
sudo chmod u+s /bin/cat
```

If `cat` is root-owned, this attempts to make it run with effective UID root. An ordinary user may then be able to read arbitrary root-readable files, including `/etc/shadow`.

`cat` is not designed as a secure privilege-mediation tool. It accepts arbitrary paths from the caller, so granting it root SUID is fundamentally unsafe.

The exact result can be affected by `nosuid`, containers, security modules, and filesystem behavior, but the configuration is still dangerous.

To remove SUID:

```bash
sudo chmod u-s /bin/cat
```

Never alter a real system binary for practice.

## 16. Applications of SUID

Legitimate SUID programs expose a **small, controlled privileged action** without giving the user unrestricted root access.

Possible uses include:

- changing one's password,
- selected authentication helpers,
- some legacy mount/network tools,
- narrowly scoped administrative utilities.

A secure SUID program should validate inputs, avoid shells, use fixed paths, drop privileges quickly, and expose only the intended action.

## 17. SUID and `passwd`

`passwd` is the classic example. Users must change their own passwords, but must not directly edit protected password files.

A typical executable may appear as:

```text
-rwsr-xr-x root root /usr/bin/passwd
```

When a user runs `passwd`, the program can perform the protected update, but it does not provide an unrestricted root shell. It implements a controlled workflow.

In [ ]:
command -v passwd
ls -l "$(command -v passwd)"
stat -c '%A %a %U %G %n' "$(command -v passwd)"

## 18. Finding SUID files

```bash
find / -xdev -type f -perm -4000 2>/dev/null
find / -xdev -type f -perm -4000 -exec ls -l {} \; 2>/dev/null
```

Explanation:

- `-type f`: regular files
- `-perm -4000`: SUID bit is set
- `-xdev`: remain on the current filesystem
- `2>/dev/null`: suppress permission-denied messages
- `-exec ls -l`: show details

This is useful for audits, detecting unexpected privileged binaries, hardening, incident response, and comparing against a trusted baseline. Not every SUID file is malicious; determine whether it is expected, trusted, necessary, correctly owned, and patched.

# Part III — SGID

## 19. What is SGID?

**SGID** means **Set Group ID**.

On an executable file, the process normally uses the file's group as its effective GID.

On a directory, newly created files and subdirectories normally inherit the directory's group instead of the creator's primary group.

## 20. Enable, disable, and inspect SGID

```bash
chmod g+s object       # enable
chmod g-s object       # disable
chmod 2755 object      # numeric enable
chmod 0755 object      # numeric disable while retaining 755
```

Inspect:

```bash
ls -l object
stat -c '%A %a %U %G %n' object
```

Examples:

```text
-rwxr-sr-x  SGID executable
-rwxr-Sr-x  SGID set, group execute missing
drwxrwsr-x  SGID directory
```

## 21. Is SGID useful on every type of file?

Its practical behavior is mainly useful on:

- executable binary files,
- directories.

It has little practical effect on ordinary non-executable data files. Linux normally ignores set-ID behavior on scripts. The `nosuid` mount option may disable executable SGID behavior.

## 22. SGID on an executable file

A SGID executable runs with the executable file group's effective GID. This can provide controlled access to group-protected resources.

Conceptual example:

```text
-rwxr-sr-x root reports report_reader
-rw-r----- root reports confidential-report
```

A user could run `report_reader`, which uses group `reports` only to perform a narrow approved operation. A safe SGID program must reject arbitrary paths, validate input, avoid shells, and drop privileges when finished.

Real examples vary by distribution. Utilities such as `wall`, `write`, or legacy mail tools have historically used SGID, but modern systems may use other mechanisms.

## 23. SGID on directories: the most common practical use

Suppose a project directory belongs to group `developers`:

```bash
sudo chown root:developers /srv/project
sudo chmod 2775 /srv/project
```

It becomes:

```text
drwxrwsr-x root developers /srv/project
```

New files and subdirectories normally inherit group `developers`. Without SGID, each user's primary group might be used, creating inconsistent ownership.

## 24. Concrete collaborative example

```bash
sudo groupadd developers
sudo usermod -aG developers alice
sudo usermod -aG developers bob
sudo mkdir -p /srv/project
sudo chown root:developers /srv/project
sudo chmod 2775 /srv/project
```

After Alice creates a file, it normally inherits group `developers`. Bob's new subdirectory also normally inherits that group, and often the SGID bit as well.

Users may need to log out and back in after group membership changes.

## 25. SGID does not automatically provide group write permission

SGID controls **group inheritance**, not all final permission bits.

The result also depends on:

- the program's requested creation mode,
- `umask`,
- default ACLs,
- filesystem behavior.

With `umask 0022`, a file may still be `0644`. Collaborative projects often use `umask 0002` or carefully configured default ACLs.

Example default ACL approach:

```bash
sudo setfacl -d -m g:developers:rwx /srv/project
sudo setfacl -d -m o::rx /srv/project
```

## 26. Finding SGID files and directories

```bash
find / -xdev -type f -perm -2000 2>/dev/null
find / -xdev -type d -perm -2000 2>/dev/null
find / -xdev -perm -2000 -exec ls -ld {} \; 2>/dev/null
```

These commands help find privileged executables, collaborative directories, and unexpected configurations.

# Part IV — Comparison and Safe Practice

## 27. Comparison table

| Bit | Octal | Object | Effect |
|---|---:|---|---|
| SUID | `4000` | executable | use file owner's effective UID |
| SGID | `2000` | executable | use file group's effective GID |
| SGID | `2000` | directory | inherit directory group |
| Sticky | `1000` | directory | restrict deletion/renaming in shared writable directory |

Typical modes:

| Mode | Meaning |
|---:|---|
| `4755` | SUID executable |
| `2755` | SGID executable/directory |
| `2775` | collaborative SGID directory |
| `1777` | shared sticky directory such as `/tmp` |
| `6755` | SUID + SGID executable |

## 28. Security checklist

Ask:

1. Is this SUID/SGID executable expected and necessary?
2. Is it owned by the correct user/group?
3. Can an untrusted user modify it or its parent directory?
4. Is the package current and trusted?
5. Can the special bit be removed?
6. Would `nosuid` be appropriate on the filesystem?
7. Does a publicly writable directory have the sticky bit?
8. Does a collaborative SGID directory have a suitable `umask` or ACL?
9. Are results compared with a known-good baseline?

A privileged binary writable by an untrusted user is a critical vulnerability.

## 29. Safe lab: sticky and SGID directories

The following commands affect only directories under the current working directory and do not require root.

In [ ]:
rm -rf special_bits_lab
mkdir -p special_bits_lab/public special_bits_lab/team
chmod 1777 special_bits_lab/public
chmod 2775 special_bits_lab/team
ls -ld special_bits_lab/public special_bits_lab/team

touch special_bits_lab/team/example.txt
ls -l special_bits_lab/team/example.txt
stat -c '%A %a %U %G %n' special_bits_lab/public special_bits_lab/team

## 30. Read-only audit lab

These commands only inspect existing files:

In [ ]:
echo "SUID files on current filesystem:"
find / -xdev -type f -perm -4000 -exec ls -l {} \; 2>/dev/null | head -n 30

echo
echo "SGID files on current filesystem:"
find / -xdev -type f -perm -2000 -exec ls -l {} \; 2>/dev/null | head -n 30

## 31. Commands not to use on a real system

```bash
sudo chmod u+s /bin/cat
sudo chmod u+s /bin/bash
sudo chmod u+s /usr/bin/python3
sudo chmod 777 /etc/shadow
sudo chown youruser /etc/passwd
```

These can create privilege-escalation paths or damage system security.

# Part V — Questions and Answers

## 32. Review questions

1. What does the leading `1` mean in `1777`?
2. Why does `/tmp` use `1777` instead of `0777`?
3. Who may normally delete a file inside a sticky directory?
4. Does sticky prevent the owner from editing the file?
5. What does SUID stand for?
6. Where does SUID appear in `ls -l`?
7. What is the difference between `s` and `S`?
8. Why is root-SUID `cat` dangerous?
9. Why is `passwd` a legitimate SUID example?
10. Is SUID on Linux shell scripts normally honored?
11. What does SGID do on an executable?
12. What does SGID do on a directory?
13. Does SGID guarantee group write permission?
14. Why use mode `2775` for a project directory?
15. Difference between `chmod g+s dir` and `chmod +t dir`?
16. What does `find / -xdev -type f -perm -4000` find?
17. Why is `nosuid` useful?
18. Is a separate `/tmp` mandatory?
19. Why can `noexec` break applications?
20. Why should SUID/SGID programs be small and audited?

## 33. Answers

1. It enables the sticky bit.
2. Sticky prevents users from deleting/renaming one another's entries.
3. Normally the file owner, directory owner, or root.
4. No; it mainly controls deletion and renaming.
5. Set User ID.
6. In the owner's execute position.
7. `s` includes execute; `S` means the execute bit is missing.
8. It may read arbitrary root-readable files.
9. It gives a controlled method to update protected password data.
10. Normally no; Linux generally ignores set-ID bits on interpreted scripts.
11. It uses the file group's effective GID.
12. New entries inherit the directory's group.
13. No; `umask`, ACLs, and requested modes still matter.
14. It supports group collaboration and inherited group ownership while denying writes to others.
15. `g+s` enables group inheritance; `+t` restricts deletion/renaming.
16. SUID regular files on the current filesystem.
17. It disables SUID/SGID execution behavior on that filesystem.
18. No; it is a design/hardening choice.
19. Some installers, build tools, and runtimes execute temporary helpers.
20. Any bug in privileged code may become a privilege-escalation vulnerability.

## 34. Final cheat sheet

```text
SUID   = 4000 = owner's effective identity during execution
SGID   = 2000 = group's effective identity / directory group inheritance
Sticky = 1000 = protected deletion/renaming in shared directories
```

```bash
chmod u+s file       # enable SUID
chmod u-s file       # disable SUID
chmod g+s object     # enable SGID
chmod g-s object     # disable SGID
chmod +t directory   # enable sticky
chmod -t directory   # disable sticky
```

```bash
find / -xdev -type f -perm -4000 2>/dev/null
find / -xdev -type f -perm -2000 2>/dev/null
find / -xdev -type d -perm -2000 2>/dev/null
```

Most important principle:

> Grant the smallest necessary privilege through carefully controlled files and directories.